# 🧬 Cell Nuclei Segmentation using U-Net

This notebook uses the **2018 Data Science Bowl nuclei segmentation dataset**. The dataset contains microscopy images and nucleus masks, making it suitable for a U-Net segmentation project. Dataset source: Kaggle 2018 Data Science Bowl.

Run the cells from top to bottom. A Kaggle API token is required to download the dataset automatically.

In [ ]:
!pip -q install kaggle tensorflow opencv-python scikit-learn

In [ ]:
from google.colab import files
files.upload()

# Upload your kaggle.json when prompted.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!mkdir -p /content/data
!kaggle competitions download -c data-science-bowl-2018 -p /content/data
!unzip -q /content/data/stage1_train.zip -d /content/data/stage1_train

In [ ]:
import os, glob, cv2, numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '/content/data/stage1_train'
image_dirs = sorted(glob.glob(os.path.join(DATA_DIR, '*', 'images', '*')))
print('Images found:', len(image_dirs))

In [ ]:
IMG_SIZE = 128
X, Y = [], []

for image_path in image_dirs:
    folder = os.path.dirname(os.path.dirname(image_path))
    mask_paths = glob.glob(os.path.join(folder, 'masks', '*.png'))

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = np.maximum(mask, m)

    X.append(image / 255.0)
    Y.append((mask > 0).astype(np.float32)[..., None])

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)
print('X shape:', X.shape)
print('Y shape:', Y.shape)

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(X[0])
plt.title('Input Image')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(Y[0].squeeze(), cmap='gray')
plt.title('Ground Truth Mask')
plt.axis('off')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
print('Training:', X_train.shape)
print('Validation:', X_val.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(input_shape)

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    u5 = layers.UpSampling2D()(c4)
    u5 = layers.Concatenate()([u5, c3])
    c5 = conv_block(u5, 128)

    u6 = layers.UpSampling2D()(c5)
    u6 = layers.Concatenate()([u6, c2])
    c6 = conv_block(u6, 64)

    u7 = layers.UpSampling2D()(c6)
    u7 = layers.Concatenate()([u7, c1])
    c7 = conv_block(u7, 32)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return Model(inputs, outputs)

model = build_unet()
model.summary()

In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.reshape(y_pred, [-1])
    y_true = tf.reshape(y_true, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

model.compile(optimizer='adam', loss=dice_loss, metrics=['accuracy', dice_coef])

In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=10,
    batch_size=8
)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('U-Net Training')
plt.legend()
plt.show()

In [ ]:
pred = model.predict(X_val[:5])

plt.figure(figsize=(12,8))
for i in range(5):
    plt.subplot(3,5,i+1)
    plt.imshow(X_val[i])
    plt.title('Input')
    plt.axis('off')

    plt.subplot(3,5,i+6)
    plt.imshow(Y_val[i].squeeze(), cmap='gray')
    plt.title('True Mask')
    plt.axis('off')

    plt.subplot(3,5,i+11)
    plt.imshow((pred[i].squeeze() > 0.5), cmap='gray')
    plt.title('Prediction')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
results = model.evaluate(X_val, Y_val, verbose=0)
print('Validation Loss:', results[0])
print('Validation Accuracy:', results[1])
print('Validation Dice:', results[2])

In [ ]:
model.save('/content/unet_cell_segmentation.keras')
print('Model saved successfully.')